# 10 · Governance dei subagenti

Quando un agente **delega** lavoro ad altri agenti (subagenti), non basta fidarsi del
nome scelto dal modello. Serve un controllo automatico: chi può fare cosa, con quali
strumenti, in che ordine, e con quali prove di fine lavoro.

In questo notebook il **router** (qui simulato a mano) propone un piano; un
**validatore deterministico** — codice Python, non un LLM — applica quattro regole:

1. **least privilege**: assegna solo un profilo che ha davvero le capability e i tool richiesti;
2. **dipendenze acicliche**: i task non possono dipendere l’uno dall’altro in un ciclo;
3. **review indipendente**: chi ha costruito non può certificare sé stesso;
4. **protocollo di risultato**: `complete` senza evidenze non vale.

Tutto gira offline, con sola standard library. Nessuna chiamata API.

## Obiettivi, prerequisiti e modalità di lettura

Alla fine di questo notebook saprai:

- leggere un **profilo** (cosa un agente può fare) e un **task** (cosa serve fare);
- capire perché un'assegnazione sbagliata viene rifiutata o corretta;
- riconoscere un **ciclo di dipendenze** (DAG = grafo diretto aciclico);
- distinguere una risposta "plausibile" da un esito **verificabile** con prove.

**Durata:** 25–35 minuti. **Prerequisiti:** aver visto i notebook su planning/subagenti
(04) e harness (05). **Modalità:** tutto offline, senza modello. Se un `assert` fallisce,
rileggi l'output atteso prima di riavviare il kernel.

Ogni blocco di codice è preceduto da una spiegazione e seguito da un **output
atteso**. Quando interviene un modello, l'output atteso descrive proprietà e
invarianti, non una frase letterale. Esegui le celle in ordine e non saltare i
casi negativi: mostrano il confine del meccanismo, non un incidente del corso.

## 1 · Profili e task sono contratti

Prima di validare un piano, servono due tipi di dato: **chi esiste** nel sistema
(`Profile`) e **cosa va fatto** (`Task`). Entrambi sono contratti espliciti, non
etichette decorative.

### Spiegazione del blocco · Profili e task

**Problema che risolviamo.** Se il piano dice solo «manda il lavoro a `researcher`»,
il runtime non sa se quel nome può davvero scrivere file o usare la sandbox. Il nome
è un'etichetta; i permessi devono stare nei dati.

**Cosa definisce il blocco.**

- `Profile`: un agente del roster. Ha `capabilities` (es. `research`, `python`),
  `tools` (es. `browser`, `sandbox`) e `read_only` (se `True` non può scrivere).
- `Task`: un pezzo di lavoro. Dichiaro **requisiti** (`capabilities`, `tools`,
  `writes`), una scelta preferita (`agent`) e alternative (`alternatives`), più
  eventuali `depends_on` (altri task da completare prima).

**Cosa osservare nel roster.** Tre ruoli diversi:

| Nome | Può | Tool | Scrive? |
|------|-----|------|---------|
| `researcher` | research | browser | no |
| `builder` | python | sandbox | sì |
| `reviewer` | review | — | no |

Il messaggio didattico: il router **propone** un nome; il validatore **verifica**
contro questi campi. Non si deduce «`builder` suona come chi scrive codice» dal nome.

In [ ]:
from dataclasses import dataclass, field

# Profile = "chi è questo agente e cosa gli è permesso".
# frozen=True: dopo la creazione non si possono cambiare i permessi a runtime.
@dataclass(frozen=True)
class Profile:
    name: str
    capabilities: set[str]   # competenze dichiarate (es. {"python"})
    tools: set[str]          # strumenti ammessi (es. {"sandbox"})
    read_only: bool          # True = non può produrre side-effect in scrittura

# Task = "cosa serve fare". I requisiti stanno qui, non nel prompt.
@dataclass
class Task:
    id: str
    agent: str                      # scelta preferita dal router
    alternatives: list[str]         # candidati se la preferita non è valida
    capabilities: set[str]          # capability richieste dal lavoro
    tools: set[str]                 # tool richiesti
    writes: bool                    # True se il task deve scrivere file / mutare stato
    kind: str = "work"              # es. "work" oppure "review"
    depends_on: list[str] = field(default_factory=list)  # id di altri task

# Roster: l'unico catalogo di agenti che il validatore conosce.
roster = {
    "researcher": Profile("researcher", {"research"}, {"browser"}, True),
    "builder": Profile("builder", {"python"}, {"sandbox"}, False),
    "reviewer": Profile("reviewer", {"review"}, set(), True),
}

### Output atteso

Nessun output a schermo. Dopo l'esecuzione esistono le classi `Profile` e `Task`,
e il dizionario `roster` con tre chiavi: `researcher`, `builder`, `reviewer`.
Puoi ispezionare con `roster.keys()` o `roster["builder"]` in una cella temporanea.

## 2 · Validare capability, tool e scrittura

Ora colleghiamo profilo e task. La regola si chiama **least privilege**: non basta
che un agente “possa fare qualcosa di simile”; deve soddisfare **tutti** i requisiti,
e preferiamo non assegnare piuttosto che assegnare in violazione.

### Spiegazione del blocco · Assegnazione least-privilege

**Idea centrale.** `supports(profile, task)` risponde sì solo se:

1. ogni capability del task è nel profilo (`task.capabilities <= profile.capabilities`);
2. ogni tool del task è nel profilo;
3. se il task deve scrivere (`writes=True`), il profilo **non** è `read_only`.

`assign` prova prima `task.agent`, poi le `alternatives`, e aggiorna `task.agent`
solo quando trova un candidato compatibile. Se nessuno va bene, restituisce `None`
(nessuna delega silenziosa).

**Scenario di questo blocco (volutamente "sbagliato").** Il router propone
`researcher` per un task Python che usa `sandbox` e scrive. `researcher` è
read-only e non ha capability `python`: fallisce. L'alternativa `builder` invece
passa tutti i controlli. Vedrai stampato `Assegnato a: builder`.

Questo è il punto del notebook: il modello può sbagliare la proposta; il codice
corregge (o rifiuta) in modo osservabile.

In [ ]:
def supports(profile: Profile, task: Task) -> bool:
    # `<=` tra set: ogni elemento a sinistra deve stare a destra (sottoinsieme).
    stessi_permessi = (
        task.capabilities <= profile.capabilities
        and task.tools <= profile.tools
    )
    # Un profilo read-only non può ricevere un task che scrive.
    scrittura_ok = not (task.writes and profile.read_only)
    return stessi_permessi and scrittura_ok


def assign(task: Task) -> Task | None:
    # Ordine: preferita del router, poi alternative. Prima compatibile vince.
    for name in [task.agent, *task.alternatives]:
        if name in roster and supports(roster[name], task):
            task.agent = name
            return task
    return None  # meglio nessuna assegnazione che una violazione


# Router "sbagliato": propone researcher per lavoro Python mutativo.
task = Task("build", "researcher", ["builder"], {"python"}, {"sandbox"}, True)
validated = assign(task)
assert validated and validated.agent == "builder"
print("Assegnato a:", validated.agent)

### Output atteso

```text
Assegnato a: builder
```

Gli `assert` passano in silenzio. Se vedi un `AssertionError`, probabilmente non hai
eseguito la cella del roster, oppure hai modificato i profili.

## 3 · Bloccare cicli e review non indipendenti

Due controlli diversi, spesso confusi:

1. **ordine dei task** — le dipendenze devono formare un grafo senza cicli (DAG);
2. **indipendenza della review** — chi ha prodotto il lavoro non può essere anche il reviewer.

### Spiegazione del blocco · DAG e review indipendente

**Parte A — cicli.** Se il task `a` dipende da `b` e `b` dipende da `a`,
l'orchestratore non sa da dove partire: è un deadlock logico. `has_cycle` fa una
visita in profondità (DFS): se mentre esplora un nodo lo reincontra sulla
catena corrente (`visiting`), c'è un ciclo. Il primo esempio costruisce apposta
`a ↔ b` e verifica che venga rilevato.

**Parte B — review indipendente.** Creiamo un task di tipo `review` che dipende
da `build`. Il router propone di nuovo `builder` (chi ha costruito), ma il
validatore, grazie alle alternative e alla capability `review`, assegna
`reviewer`. L'assert finale controlla anche `review.agent != validated.agent`:
non basta avere un nome "reviewer" nel piano; deve essere **un agente diverso**
da chi ha prodotto l'artefatto. Altrimenti è autocertificazione.

In [ ]:
def has_cycle(tasks: list[Task]) -> bool:
    # Grafo: id del task -> lista di id da cui dipende.
    graph = {task.id: task.depends_on for task in tasks}
    visiting, done = set(), set()

    def visit(node):
        if node in visiting:
            return True   # lo stiamo ancora esplorando: ciclo trovato
        if node in done:
            return False  # già esplorato senza cicli
        visiting.add(node)
        if any(visit(dep) for dep in graph.get(node, [])):
            return True
        visiting.remove(node)
        done.add(node)
        return False

    return any(visit(node) for node in graph)


# Caso negativo: a dipende da b e b da a → ciclo.
cycle = [
    Task("a", "builder", [], set(), set(), False, depends_on=["b"]),
    Task("b", "builder", [], set(), set(), False, depends_on=["a"]),
]
assert has_cycle(cycle)

# Review: il router propone builder (chi ha costruito), ma serve capability "review".
review = Task(
    "review", "builder", ["reviewer"], {"review"}, set(), False,
    kind="review", depends_on=["build"],
)
review = assign(review)
assert review and review.agent == "reviewer" and review.agent != validated.agent
print("Reviewer indipendente:", review.agent)

### Output atteso

```text
Reviewer indipendente: reviewer
```

Gli `assert` sul ciclo e sull'indipendenza passano senza stampare nulla.
Se `validated` non esiste, riesegui la sezione 2 nello stesso kernel.

## 4 · Protocollo di risultato

Finora abbiamo governato **chi** lavora e **in che ordine**. Manca il contratto di
uscita: quando un subagente dice «ho finito», cosa deve consegnare perché il
sistema possa credergli?

### Spiegazione del blocco · Protocollo terminale

**Problema.** Un subagente può restituire una frase fluida («tutto ok, test passati»)
senza allegare nulla di verificabile. In un harness serio quello non basta.

**Contratto minimale di questo blocco.**

- `status`: esito dichiarato (qui usiamo `"complete"`);
- `evidence`: prove osservabili (es. riga di log di pytest);
- `artifacts`: percorsi o nomi di file prodotti.

`terminal_report` **rifiuta** `status="complete"` se `evidence` è vuota: solleva
`ValueError`. Evidenze e artefatti non sono opzionali nel caso di successo; sono
parte del messaggio di fine lavoro. (Prova a chiamare
`terminal_report("complete", [], [])` in una cella a parte: deve fallire.)

In [ ]:
def terminal_report(status: str, evidence: list[str], artifacts: list[str]) -> dict:
    # Successo dichiarato senza prove = contratto violato.
    if status == "complete" and not evidence:
        raise ValueError("complete richiede evidenza")
    return {"status": status, "evidence": evidence, "artifacts": artifacts}


# Caso valido: stato + prova + artefatto.
report = terminal_report("complete", ["pytest: 12 passed"], ["report.md"])
print(report)

### Output atteso

Qualcosa di equivalente a:

```text
{'status': 'complete', 'evidence': ['pytest: 12 passed'], 'artifacts': ['report.md']}
```

Se chiami `terminal_report("complete", [], [])` ottieni `ValueError: complete richiede evidenza`.
Quello è il comportamento corretto, non un bug.

## Prova tu

Estendi il modello con un **budget per agente** (es. token o “unità di lavoro” residue).

1. Aggiungi a `Profile` (o a un dizionario parallelo) un campo `budget` iniziale.
2. Prima di ogni invocazione, controlla se il residuo è ≥ costo stimato della chiamata.
3. Se il residuo non basta nemmeno per una risposta finale, **impedisci** una seconda
   invocazione e restituisci uno stato terminale esplicito (es. `budget_exceeded`),
   non un silenzioso “riprova”.

Obiettivo: capire che anche una delega valida può fallire per limiti di risorsa,
e che il limite deve essere codice, non fiducia nel modello.

## Laboratorio aggiuntivo

Gli esempi seguenti riusano `assign` e `has_cycle` definiti sopra: **non riavviare**
il kernel a meno che non sia necessario. Il primo caso mostra un rifiuto netto
(nessun agente compatibile); il secondo un ciclo più lungo di due nodi, tipico
bug quando le dipendenze vengono generate automaticamente.

## Esempio aggiuntivo: nessun agente compatibile

### Spiegazione del blocco

**Perché importa.** In produzione è tentante "forzare" un'assegnazione comunque
(es. dare il task al primo agente disponibile). Qui facciamo l'opposto: se nessuno
nel roster soddisfa i requisiti, `assign` restituisce `None`.

**Lettura del task.** `deploy` chiede capability `python`, tool `sandbox` e
`writes=True`. Candidati: `researcher` (read-only, no python) e `reviewer`
(no python, no sandbox). Nessuno passa `supports` → nessuna delega.

Messaggio da ricordare: **fallire in modo esplicito** è meglio di una delega
che viola least privilege.

In [ ]:
# Nessun candidato nel roster può: python + sandbox + scrittura.
impossibile = Task("deploy", "researcher", ["reviewer"], {"python"}, {"sandbox"}, True)
print("Assegnazione:", assign(impossibile))

### Output atteso

```text
Assegnazione: None
```

`None` qui è successo didattico: il validatore ha rifiutato invece di forzare.
Se stampasse un nome agente, avresti una violazione di least privilege.

## Esempio aggiuntivo: ciclo indiretto

### Spiegazione del blocco

**Scenario.** Tre task:

- `a` dipende da `b`
- `b` dipende da `c`
- `c` dipende da `a`

Nessuna freccia è un auto-loop a due nodi, ma insieme formano un ciclo.
`has_cycle` deve percorrere tutto il grafo (DFS / visite) e restituire `True`.

In un planner generato da LLM questo è frequente: ogni passo "sembra" locale e
ragionevole, ma l'insieme non è eseguibile. Il validatore deve vedere il piano
intero, non solo i vicini di un nodo.

In [ ]:
# a → b → c → a: ciclo a tre nodi (indiretto).
ciclo_lungo = [
    Task("a", "builder", [], set(), set(), False, depends_on=["b"]),
    Task("b", "builder", [], set(), set(), False, depends_on=["c"]),
    Task("c", "reviewer", [], set(), set(), False, depends_on=["a"]),
]
print("Ciclo rilevato:", has_cycle(ciclo_lungo))

### Output atteso

```text
Ciclo rilevato: True
```

Se ottenessi `False`, il rilevatore starebbe controllando solo archi diretti
e perderebbe i cicli lunghi — bug tipico da evitare in produzione.

## Riepilogo e troubleshooting

**Cosa hai visto in quattro punti.**

1. **Contratti espliciti** — profilo e task dichiarano capability, tool e scrittura.
2. **Least privilege** — il validatore corregge o rifiuta la proposta del router.
3. **DAG + review indipendente** — niente cicli; chi costruisce non autocertifica.
4. **Protocollo terminale** — `complete` senza evidenze non è accettabile.

Prima di passare al notebook successivo, prova a spiegare a voce: *chi* ha deciso
l'assegnazione (codice, non modello), *perché* `researcher` è stato scartato, e
*cosa* rende osservabile un esito `complete`.

Se una cella fallisce:

1. rileggi l'output atteso e individua la prima invariante non rispettata;
2. verifica di aver eseguito tutte le celle precedenti **nello stesso kernel**;
3. questo notebook è offline: non serve `.env` né quota API;
4. riavvia il kernel solo dopo aver annotato cosa stavi ispezionando;
5. non "correggere" un caso negativo (`None`, ciclo, `ValueError`): è parte della lezione.